# Phase 5: Classification -- Demand Risk Regimes

## Why this is a genuinely different task from Phase 4

Phase 4 asked "exactly how many MW will this hour need." This phase asks a
narrower, more operational question: "is this upcoming hour Low, Normal,
Peak, or Critical demand?" A grid operator often cares more about the
second question -- it's what actually triggers real decisions like
preparing demand-response programs -- and it comes with genuinely
different success criteria than MAE/MAPE: precision, recall, and the
trade-off between them, especially given that the category we care about
catching most (Critical) will also be the rarest by construction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

pd.set_option("display.max_columns", None)

train_df = pd.read_csv('../data/processed/train.csv', parse_dates=['datetime'])
test_2023 = pd.read_csv('../data/processed/test_2023.csv', parse_dates=['datetime'])
test_2024 = pd.read_csv('../data/processed/test_2024.csv', parse_dates=['datetime'])
test_2025 = pd.read_csv('../data/processed/test_2025.csv', parse_dates=['datetime'])

TARGET_COL = 'ontario_demand'
EXCLUDE_COLS = {
    'datetime', TARGET_COL, 'market_demand', 'hoep', 'temperature_c',
    'year_source', 'days_since_start', 'Date', 'Hour', 'hour_of_day', 'month', 'year',
}
feature_cols = [c for c in train_df.columns if c not in EXCLUDE_COLS]
print(f"Reusing {len(feature_cols)} features from Phase 3/4, unchanged -- only the target changes.")

## 1. Defining the regime categories -- thresholds from TRAINING data only

Using demand percentiles to define four categories: Low (bottom 25%),
Normal (25th-75th percentile), Peak (75th-95th), and Critical (top 5%).
This is a reasonable, defensible split for this project's scope -- a more
elaborate version could instead use IESO's own published demand-response
trigger thresholds, worth naming as a documented simplification rather
than presenting the percentile cutoffs as if they were official grid
operator thresholds.

**Critical leakage point:** the percentile thresholds themselves must be
computed from TRAINING data only, then applied unchanged to the test
years. Computing thresholds from the full dataset (including 2023-2025)
would let information about the test period's actual distribution leak
into how categories are defined in the first place -- a subtler form of
the same leakage discipline from every prior phase.

In [ ]:
quantiles = train_df[TARGET_COL].quantile([0.25, 0.75, 0.95])
LOW_THRESHOLD, PEAK_THRESHOLD, CRITICAL_THRESHOLD = quantiles[0.25], quantiles[0.75], quantiles[0.95]

print(f"Low:      demand < {LOW_THRESHOLD:.0f} MW")
print(f"Normal:   {LOW_THRESHOLD:.0f} - {PEAK_THRESHOLD:.0f} MW")
print(f"Peak:     {PEAK_THRESHOLD:.0f} - {CRITICAL_THRESHOLD:.0f} MW")
print(f"Critical: demand > {CRITICAL_THRESHOLD:.0f} MW")


def assign_regime(demand_series):
    return pd.cut(
        demand_series,
        bins=[-np.inf, LOW_THRESHOLD, PEAK_THRESHOLD, CRITICAL_THRESHOLD, np.inf],
        labels=['Low', 'Normal', 'Peak', 'Critical'],
    )

train_df['regime'] = assign_regime(train_df[TARGET_COL])
test_2023['regime'] = assign_regime(test_2023[TARGET_COL])
test_2024['regime'] = assign_regime(test_2024[TARGET_COL])
test_2025['regime'] = assign_regime(test_2025[TARGET_COL])

train_df['regime'].value_counts()

In [ ]:
fig, ax = plt.subplots()
train_df['regime'].value_counts().reindex(['Low', 'Normal', 'Peak', 'Critical']).plot(kind='bar', ax=ax)
plt.title('Class Distribution in Training Data')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

As expected, Low and Normal dominate (this is exactly how percentile-based
thresholds work by construction) and Critical is rare. This imbalance is
the whole reason accuracy alone would be a misleading metric here -- a
model that always predicted "Normal" would still score close to 50%
accuracy while being completely useless for the thing we actually care
about: catching Critical hours.

## 2. Handling class imbalance

Rather than letting the model implicitly learn "just guess Normal most of
the time" (which minimizes overall error given how rare Critical is),
`compute_sample_weight('balanced', ...)` assigns higher weight to
under-represented classes during training -- effectively telling the model
"a mistake on a rare Critical hour costs more than a mistake on a common
Normal hour," which better matches how a real grid operator would actually
value these errors.

In [ ]:
xgb_fit_df = train_df[train_df['datetime'] < '2022-01-01']
xgb_val_df = train_df[train_df['datetime'] >= '2022-01-01']

sample_weights = compute_sample_weight('balanced', xgb_fit_df['regime'])

clf = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    early_stopping_rounds=30,
    random_state=42,
    eval_metric='mlogloss',
)

clf.fit(
    xgb_fit_df[feature_cols], xgb_fit_df['regime'].cat.codes,
    sample_weight=sample_weights,
    eval_set=[(xgb_val_df[feature_cols], xgb_val_df['regime'].cat.codes)],
    verbose=False,
)

print(f"Best iteration: {clf.best_iteration}")

## 3. A simple baseline for comparison

Same principle as comparing XGBoost against an LSTM in Phase 4: a result
only means something in contrast to a simpler alternative. Logistic
regression, with the same class balancing, is a reasonable, fast baseline
here.

In [ ]:
baseline = LogisticRegression(max_iter=1000, class_weight='balanced')
baseline.fit(xgb_fit_df[feature_cols], xgb_fit_df['regime'].cat.codes)

## 4. Evaluation -- precision/recall/F1 per class, not just accuracy

`classification_report` breaks results down by class, which is exactly
what accuracy alone can't do. Pay particular attention to Critical's
RECALL specifically -- of all the truly Critical hours, what fraction did
the model actually catch? Missing a real Critical hour (a false negative)
is generally more costly in a real grid-operations context than a false
alarm on a Peak hour, so recall on the rare class is arguably the single
most important number in this whole table.

In [ ]:
regime_labels = ['Low', 'Normal', 'Peak', 'Critical']

def evaluate_classifier(model, X, y_true_codes, label, is_baseline=False):
    preds_codes = model.predict(X)
    y_true = pd.Categorical.from_codes(y_true_codes, categories=regime_labels)
    y_pred = pd.Categorical.from_codes(preds_codes, categories=regime_labels)

    print(f"=== {label} ===")
    print(classification_report(y_true, y_pred, labels=regime_labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=regime_labels)
    disp = ConfusionMatrixDisplay(cm, display_labels=regime_labels)
    disp.plot(cmap='Blues')
    plt.title(f'Confusion Matrix: {label}')
    plt.tight_layout()
    plt.show()

for year_label, test_set in [('2023', test_2023), ('2024', test_2024), ('2025 (partial)', test_2025)]:
    evaluate_classifier(clf, test_set[feature_cols], test_set['regime'].cat.codes, f'XGBoost {year_label}')

In [ ]:
# Baseline comparison, 2023 only, for a quick sanity check against XGBoost
evaluate_classifier(baseline, test_2023[feature_cols], test_2023['regime'].cat.codes, 'Logistic Regression 2023')

## 5. Feature importance for the classifier

Worth checking whether the same features that mattered for regression
(Phase 4) also matter for classification -- they often overlap heavily,
but not always; a feature can matter a lot for pinning down the exact MW
value while contributing less to which broad category an hour falls into,
or vice versa.

In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 8))
importances.plot(kind='barh', ax=ax)
ax.invert_yaxis()
plt.title('XGBoost Classifier Feature Importance')
plt.tight_layout()
plt.show()

## Summary

The headline number to report isn't overall accuracy -- it's Critical
class recall specifically, since that's the category where a missed
prediction is most operationally costly. Report it plainly even if it's
not high: a classifier that catches, say, 60% of true Critical hours is a
genuine, disclosable result, and pairing it with precision shows the
real trade-off (catching more Critical hours usually means more false
alarms on Peak hours, and that trade-off is worth naming explicitly rather
than hiding behind a single accuracy number).

Next: **Phase 6**, using the demand/price forecasts from Phase 4 to
optimize a battery dispatch policy -- the point where prediction turns
into an actual decision.